# Exercício: Investigando Valores Ausentes no Clima

In [6]:
import pandas as pd

# Load the dataset
df = pd.read_csv('clima.csv')

# --- Part 1 ---
print("--- Parte 1: Diagnóstico ---")
nulos = df.isnull().sum()
print("Valores nulos por coluna:\n", nulos)
mais_nulos = nulos.idxmax()
menos_nulos = nulos.idxmin()
print(f"Variável com MAIS nulos: {mais_nulos} ({nulos[mais_nulos]})")
print(f"Variável com MENOS nulos: {menos_nulos} ({nulos[menos_nulos]})")
print("\n")

# --- Part 2 ---
print("--- Parte 2: Análise ---")
falha_umidade_ambas = len(df[df['Umidade9'].isnull() & df['Umidade15'].isnull()])
falha_umidade_qualquer = len(df[df['Umidade9'].isnull() | df['Umidade15'].isnull()])
print(f"Nulos em Umidade9 e Umidade15 ao mesmo tempo: {falha_umidade_ambas}")
print(f"Nulos em qualquer uma das Umidades: {falha_umidade_qualquer}")

falha_pressao_ambas = len(df[df['Pressao9'].isnull() & df['Pressao15'].isnull()])
falha_pressao_qualquer = len(df[df['Pressao9'].isnull() | df['Pressao15'].isnull()])
print(f"Nulos em Pressao9 e Pressao15 ao mesmo tempo: {falha_pressao_ambas}")
print(f"Nulos em qualquer uma das Pressões: {falha_pressao_qualquer}")

mediana_mintemp = df['MinTemp'].median()
nulos_mintemp = df['MinTemp'].isnull().sum()
print(f"MinTemp - Mediana: {mediana_mintemp}, Nulos: {nulos_mintemp}")
print("\n")

# --- Part 3 ---
# We will just verify the code logic and run it to be sure
mediana_temp = df['MinTemp'].median()
moda_vento = df['DirecaoVento'].mode()[0]
df_tratado = df.copy()
df_tratado['MinTemp'] = df_tratado['MinTemp'].fillna(mediana_temp)
df_tratado['DirecaoVento'] = df_tratado['DirecaoVento'].fillna(moda_vento)
print("--- Parte 3: Tratamento ---")
print("Nulos em MinTemp após tratamento:", df_tratado['MinTemp'].isnull().sum())
print("Nulos em DirecaoVento após tratamento:", df_tratado['DirecaoVento'].isnull().sum())
print("Total de linhas do dataset:", len(df))

--- Parte 1: Diagnóstico ---
Valores nulos por coluna:
 Data                     0
Localizacao              0
MinTemp               1485
MaxTemp               1261
Chuva                 3261
Evaporacao           62790
LuzSol               69835
DirecaoVento         10326
VelocidadeVento      10263
DirecaoVento9        10566
DirecaoVento15        4228
VelocidadeVento9      1767
VelocidadeVento15     3062
Umidade9              2654
Umidade15             4507
Pressao9             15065
Pressao15            15028
Nuvens9              55888
Nuvens15             59358
Temperatura9          1767
Temperatura15         3609
ChuvaHoje             3261
ChuvaAmanha           3267
dtype: int64
Variável com MAIS nulos: LuzSol (69835)
Variável com MENOS nulos: Data (0)


--- Parte 2: Análise ---
Nulos em Umidade9 e Umidade15 ao mesmo tempo: 1887
Nulos em qualquer uma das Umidades: 5274
Nulos em Pressao9 e Pressao15 ao mesmo tempo: 14804
Nulos em qualquer uma das Pressões: 15289
MinTemp - Mediana: 12.

### Parte 1 - Diagnóstico
1. Quantidade de valores nulos e Extremos:
Através do código, verificamos o seguinte cenário:

Variável com MAIS valores ausentes: LuzSol (69.835 nulos, quase 48% do dataset). Outras com muitos nulos são Evaporacao (62.790) e Nuvens15 (59.358).

Variável com MENOS valores ausentes: Data e Localizacao (0 nulos).

2. Existe alguma variável que pode comprometer mais a análise? Por quê?

Sim. Variáveis com quase 50% de dados nulos (como LuzSol e Evaporacao) comprometem muito a análise. Se você preencher esses buracos com médias ou medianas, estará "inventando" metade dos dados dessa coluna, introduzindo um viés enorme.

Além disso, a coluna ChuvaAmanha (que geralmente é o que queremos prever em modelos climáticos) possui 3.267 valores nulos. Treinar um modelo sem saber a "resposta correta" nesses dias prejudica o aprendizado do algoritmo.

### Parte 2 - Análise
3. Comparando variáveis das 9h e 15h:
Ao cruzar os dados, obtivemos os seguintes números:

Umidade: 1.887 registros não possuem medição nem às 9h nem às 15h.

Pressão: 14.804 registros não possuem dados de pressão em nenhum dos dois horários (sendo que o total de falhas isoladas em pressão é de apenas ~15.200).

Pergunta: Os valores ausentes parecem ocorrer nos mesmos registros?

Para a Pressão, sim, quase sempre! O fato de faltarem simultaneamente nos dois horários em 14.804 linhas indica fortemente que o barômetro (sensor de pressão) de certas estações estava quebrado ou desligado naqueles dias inteiros.

4. Analisando MinTemp:

Mediana: 12.0

Valores nulos: 1.485 (cerca de 1% do dataset).

Pergunta: A mediana representa bem os dados mesmo com valores ausentes?

Sim. Como apenas ~1% dos dados de MinTemp está faltando, a mediana (12.0) continua sendo uma medida excelente e altamente confiável para representar o centro dos dados, não sendo distorcida por essa ausência.

### Parte 3 - Tratamento
Aqui está como o código de tratamento foi estruturado para o seu DataFrame:

5. Estratégia escolhida:

Numéricas (ex: MinTemp): Preenchimento com a mediana.

Categóricas (ex: DirecaoVento): Preenchimento com a moda (o valor mais frequente).

6. Aplicação do Tratamento (Código Python recomendado):

In [7]:
import pandas as pd

# Carregando os dados
df = pd.read_csv('clima.csv')

# Tratando MinTemp (Numérica)
mediana_temp = df['MinTemp'].median()
df['MinTemp'] = df['MinTemp'].fillna(mediana_temp)

# Tratando DirecaoVento (Categórica)
moda_vento = df['DirecaoVento'].mode()[0]
df['DirecaoVento'] = df['DirecaoVento'].fillna(moda_vento)

# 7. Verificando os nulos após o tratamento
print("Nulos em MinTemp:", df['MinTemp'].isnull().sum()) # Retornará 0
print("Nulos em DirecaoVento:", df['DirecaoVento'].isnull().sum()) # Retornará 0

Nulos em MinTemp: 0
Nulos em DirecaoVento: 0


### Parte 4 - Desafio Final
8. Você confiaria nesse dataset para treinar um modelo de Machine Learning?

Parcialmente. Eu confiaria para variáveis como MinTemp, MaxTemp e Umidade, pois a quantidade de nulos é pequena em relação ao total de dados (145 mil linhas). Porém, não confiaria em usar colunas como LuzSol, Evaporacao ou Nuvens, a menos que elas sejam completamente descartadas (dropadas) do treinamento devido ao altíssimo número de falhas.

9. O que ainda te preocupa?

A sazonalidade ignorada: Substituir a temperatura ou umidade pela mediana global pode ser perigoso em dados de clima. A temperatura média no inverno é muito diferente do verão. Preencher um buraco no mês de julho com a mediana do ano inteiro cria dados irreais.

A variável alvo: Os 3.267 nulos na variável ChuvaAmanha. Em machine learning, a melhor prática é excluir as linhas em que a variável que você quer prever está vazia, em vez de tentar adivinhar se choveu ou não.